In [1]:
import cytnx
import numpy as np

In [2]:
path = './Networks'

def ut_print(ut, print_numpy=True):
    ut.print_diagram()
    if print_numpy:
        print(ut.get_block().numpy())

def get_M(T):
    W = np.array([[np.exp(1/T), np.exp(-1/T)],
                  [np.exp(-1/T), np.exp(1/T)]])
    W = cytnx.from_numpy(W)
    S, U, Vd = cytnx.linalg.Svd(W)
    M = U @ cytnx.linalg.Diag(S.Pow(0.5))
    M = cytnx.UniTensor(M, rowrank=1)
    M.set_name('M')
    Md = cytnx.linalg.Diag(S.Pow(0.5)) @ Vd
    Md = cytnx.UniTensor(Md, rowrank=1)
    Md.set_name('Md')
    return M, Md

def get_delta(h):
    delta = cytnx.zeros([2, 2, 2, 2])
    delta[0, 0, 0, 0] = 1
    delta[1, 1, 1, 1] = 1
    delta = cytnx.UniTensor(delta, rowrank=2)
    delta.set_name('delta')
    return delta

def get_T_baret(T, h=0):
    ut_M, ut_Md = get_M(T)
    ut_delta = get_delta(h)
    Ising_net = cytnx.Network(f'{path}/Ising_square.net')
    Ising_net.PutUniTensors(['delta', 'M0.d', 'M1.d', 'M2', 'M3'],
                            [ut_delta, ut_Md, ut_Md, ut_M, ut_M])
    T_bare = Ising_net.Launch()
    T_bare.set_name('T_bare')
    return T_bare

def merge_y(Tup, Tdn, combine=False, trace=False):
    if trace:
        TupTdn_net = cytnx.Network(f'{path}/merge_y_trace.net')
    else:
        TupTdn_net = cytnx.Network(f'{path}/merge_y.net')
    TupTdn_net.PutUniTensors(['Tup', 'Tdn'], [Tup, Tdn])
    TupTdn = TupTdn_net.Launch()
    if combine:
        TupTdn.combineBonds([1, 2])
        TupTdn.combineBonds([3, 4])
    return TupTdn

def merge_y_truncate(Tup, Tdn, dcut):
    if ((Tup.shape()[1] * Tdn.shape()[1]) < dcut):
        return merge_y(Tup, Tdn, True)
    TupTdn_pure_net = cytnx.Network(f'{path}/merge_y_pure.net')
    TupTdn_pure_net.PutUniTensors(['Tup', 'Tdn', 'Tupd', 'Tdnd'],
                                  [Tup, Tdn, Tup, Tdn])
    TupTdn_pure = TupTdn_pure_net.Launch()
    _, U, __ = cytnx.linalg.Svd_truncate(TupTdn_pure, dcut)
    TupTdn_net = cytnx.Network(f'{path}/merge_y_truncate.net')
    TupTdn_net.PutUniTensors(['Tup', 'Tdn', 'UL', 'UR'],
                             [Tup, Tdn, U, U])
    TupTdn = TupTdn_net.Launch()
    return TupTdn

def merge_x(TL, TR, combine=False, trace=False):
    if trace:
        n = TL.shape()[0]
        I = np.eye(n)
        I = cytnx.from_numpy(I)
        I = cytnx.UniTensor(I, rowrank=1)
        TLTR_net = cytnx.Network(f'{path}/merge_x_trace.net')
        TLTR_net.PutUniTensors(['TL', 'TR', 'IL', 'IR'], [TL, TR, I, I])
        TLTR = TLTR_net.Launch()
    else:
        TLTR_net = cytnx.Network(f'{path}/merge_x.net')
        TLTR_net.PutUniTensors(['TL', 'TR'], [TL, TR])
        TLTR = TLTR_net.Launch()
    if combine and not trace:
        TLTR.combineBonds([0, 1])
        TLTR.combineBonds([4, 5])
    return TLTR

def merge_x_truncate(TL, TR, dcut):
    if ((TL.shape()[0] * TR.shape()[0]) < dcut):
        return merge_x(TL, TR, True)
    TLTR_pure_net = cytnx.Network(f'{path}/merge_x_pure.net')
    TLTR_pure_net.PutUniTensors(['TL', 'TR', 'TLd', 'TRd'],
                                [TL, TR, TL, TR])
    TLTR_pure = TLTR_pure_net.Launch()
    try:
        _, U, __ = cytnx.linalg.Svd_truncate(TLTR_pure, dcut)
    except:
        TLTR_pure.print_diagram()
        _, U, __ = cytnx.linalg.Svd(TLTR_pure, dcut)
    TLTR_net = cytnx.Network(f'{path}/merge_x_truncate.net')
    TLTR_net.PutUniTensors(['TL', 'TR', 'Uup', 'Udn'], [TL, TR, U, U])
    TLTR = TLTR_net.Launch()
    return TLTR

def trace(T):
    n = T.shape()[0]
    I = np.eye(n)
    I = cytnx.from_numpy(I)
    I = cytnx.UniTensor(I, rowrank=1)
    trace_net = cytnx.Network(f'{path}/trace.net')
    trace_net.PutUniTensors(['T', 'I'], [T, I])
    return trace_net.Launch()

# No dcut

In [3]:
MaxL = 4
Temp = np.linspace(2.26,4,100)
W = np.ones((MaxL+1,len(Temp),2))
E = np.ones((MaxL+1,len(Temp),2))
Tc = 2/np.log(1+np.sqrt(2))

In [4]:
%%time
for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp)
    TL = T_bare
    for j in range(2,MaxL+1):
        TL_Trace = merge_y(TL,TL,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y(TL,TL,True)
E = -np.log(W)
Corr_len = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 6.45 s, sys: 2.46 s, total: 8.91 s
Wall time: 1.45 s


/home/TSC0731/anaconda3/envs/cytnx-07a/lib/python3.6/site-packages/ipykernel_launcher.py:10: RuntimeWarning: divide by zero encountered in true_divide
  # Remove the CWD from sys.path while we load stuff.


# dcut = 2

In [5]:
%%time
dcut = 2

for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        #print(j)
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
E = -np.log(W)
Corr_len_2 = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 3.1 s, sys: 3.52 s, total: 6.62 s
Wall time: 1.17 s


/home/TSC0731/anaconda3/envs/cytnx-07a/lib/python3.6/site-packages/ipykernel_launcher.py:15: RuntimeWarning: divide by zero encountered in true_divide
  from ipykernel import kernelapp as app


# dcut = 4

In [6]:
%%time
dcut = 4

for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        #print(j)
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
E = -np.log(W)
Corr_len_4 = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 3.22 s, sys: 3.55 s, total: 6.77 s
Wall time: 1.26 s


/home/TSC0731/anaconda3/envs/cytnx-07a/lib/python3.6/site-packages/ipykernel_launcher.py:15: RuntimeWarning: divide by zero encountered in true_divide
  from ipykernel import kernelapp as app


# dcut = 10

In [7]:
%%time
dcut = 10

for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        #print(j)
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
E = -np.log(W)
Corr_len_10 = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 11.1 s, sys: 15.5 s, total: 26.6 s
Wall time: 3.14 s


/home/TSC0731/anaconda3/envs/cytnx-07a/lib/python3.6/site-packages/ipykernel_launcher.py:15: RuntimeWarning: divide by zero encountered in true_divide
  from ipykernel import kernelapp as app


# dcut = 20

In [8]:
%%time
dcut = 20

for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        #print(j)
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
E = -np.log(W)
Corr_len_20 = 1/(E[:,:,-2]-E[:,:,-1])

CPU times: user 5min 3s, sys: 2min 10s, total: 7min 14s
Wall time: 23.4 s


/home/TSC0731/anaconda3/envs/cytnx-07a/lib/python3.6/site-packages/ipykernel_launcher.py:15: RuntimeWarning: divide by zero encountered in true_divide
  from ipykernel import kernelapp as app


In [9]:
import pickle

data = {
    "Tc": Tc,
    "MaxL": MaxL,
    "Temp": Temp,
    "Corr_len": Corr_len,
    "Corr_len_2": Corr_len_2,
    "Corr_len_4": Corr_len_4,
    "Corr_len_10": Corr_len_10,
    "Corr_len_20": Corr_len_20
}

with open("Corr_len.pkl", "wb") as file:
    pickle.dump(data, file)

## $Max = 128\times 128$

In [10]:
MaxL = 12
dcut = 16
Temp = np.linspace(2.26,2.36,100)
W = np.ones((MaxL+1,len(Temp),2))
E = np.ones((MaxL+1,len(Temp),2))
Tc = 2/np.log(1+np.sqrt(2))

In [ ]:
%%time
for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
E = -np.log(W)
Corr_len = 1/(E[:,:,-2]-E[:,:,-1])

In [ ]:
import pickle

data = {
    "Tc": Tc,
    "MaxL": MaxL,
    "Temp": Temp,
    "dcut": dcut,
    "Corr_len": Corr_len
}

with open("Corr_len_20.pkl", "wb") as file:
    pickle.dump(data, file)

In [ ]:
MaxL = 12
dcut = 20
Temp = np.linspace(2.26,2.36,250)
W = np.ones((MaxL+1,len(Temp),2))
E = np.ones((MaxL+1,len(Temp),2))
Tc = 2/np.log(1+np.sqrt(2))

In [ ]:
%%time
for i,temp in enumerate(Temp):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        TL_Trace = merge_y(TLx,TLx,True,True)
        w,v = cytnx.linalg.Eigh(TL_Trace.get_block())
        W[j,i,:] = w.numpy()[-2:]
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
    del TL, TLx
E = -np.log(W)
Corr_len = 1/(E[:,:,-2]-E[:,:,-1])

In [ ]:
import pickle

data = {
    "Tc": Tc,
    "MaxL": MaxL,
    "Temp": Temp,
    "dcut": dcut,
    "Corr_len": Corr_len
}

with open("Corr_len_16.pkl", "wb") as file:
    pickle.dump(data, file)

# Secant_Method

In [ ]:
def f(temp, MaxL, dcut = 20):
    T_bare = get_T_baret(temp) 
    TL = T_bare
    for j in range(2,MaxL+1):
        TLx = merge_x_truncate(TL,TL,dcut)
        TL = merge_y_truncate(TLx,TLx,dcut)
        TL = TL/np.mean(np.abs(TL.get_block().numpy()))
    TL_Trace = merge_y(TLx,TLx,True,True)
    w1,_ = cytnx.linalg.Eigh(TL_Trace.get_block())
    E1 = -np.log(w1.numpy()[-2:])
    Corr_len1 = 1/(E1[-2]-E1[-1])
    TLx = merge_x_truncate(TL,TL,dcut)
    TL = merge_y_truncate(TLx,TLx,dcut)
    TL = TL/np.mean(np.abs(TL.get_block().numpy()))
    TL_Trace = merge_y(TLx,TLx,True,True)
    w2,_ = cytnx.linalg.Eigh(TL_Trace.get_block())
    E2 = -np.log(w2.numpy()[-2:])
    Corr_len2 = 1/(E2[-2]-E2[-1])
    return Corr_len1-Corr_len2

def Secant_Method(func,x0,x1,Maxiter,MaxL,dcut):
    x = [x0,x1]
    func_out = [func(x0,MaxL,dcut)]
    for i in range(Maxiter):
        if (x[-1] - x[-2]) == 0:
            break
        func_out.append(func(x[-1],MaxL,dcut))
        if (func_out[-1] - func_out[-2]) == 0:
            break
        if np.abs(func_out[-1]) <= 1e-13:
            break
        new_x = x[-1] - func_out[-1] * (x[-1] - x[-2]) / (func_out[-1] - func_out[-2])
        x.append(new_x)
        #print(i,x[-1],func_out[-1])
    return x[-1]

In [ ]:
MaxL = 22
dcuts = [4, 6, 8, 12, 16, 20]
Tc = 2/np.log(1+np.sqrt(2))
T_stars = []

In [ ]:
%%time
for dcut in dcuts:
    T_star = [2.36]
    print("dcut =",dcut)
    for i in range(2,MaxL):
        #print(i , 2**(i-1))
        T_star.append(Secant_Method(f,2.26,T_star[-1],10,i,dcut))
        print(i , 2**(i-1),T_star[-1])
    T_stars.append(T_star)
T_stars = np.array(T_stars)

In [ ]:
import pickle

data = {
    "Tc": Tc,
    "MaxL": MaxL,
    "dcuts": dcuts,
    "T_stars": T_stars,
}

with open("Secant_Method.pkl", "wb") as file:
    pickle.dump(data, file)